# Relatório FOMC – Mesa de Investimentos / Depin

Notebook para geração automática do relatório Word do FOMC.

**Fluxo:**
1. Configurar data da reunião e inputs manuais (resumo, headlines, comentários)
2. Download automático de documentos do Fed (se necessário)
3. Parse do statement (decisão, taxa, votação)
4. Projeções SEP (se reunião trimestral: Mar/Jun/Set/Dez)
5. Preview do conteúdo
6. Gerar relatório .docx final

In [1]:
# Setup + Imports
import sys
from pathlib import Path

# Adiciona o diretório raiz ao path
project_root = Path.cwd().parents[3]
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import display, Markdown

from reports.fomc.core.data_loader import (
    get_fomc_dates,
    get_meeting_documents,
    is_sep_meeting,
    get_prior_sep_projections,
    load_dots_history,
    load_dots_snapshot,
    save_sep_to_excel,
)
from reports.fomc.core.pdf_parser import (
    parse_projection_table,
    parse_statement,
    get_projection_dates,
)
from reports.fomc.core.calculations import (
    get_prior_sep_meeting_date,
    get_prior_meeting_label,
)
from reports.fomc.core.fed_scraper import (
    fetch_latest_documents,
    check_for_updates,
)
from reports.fomc.core.word_report import generate_fomc_report

print("Módulos carregados com sucesso.")

Módulos carregados com sucesso.


In [2]:
# ============================================================
# CONFIGURAÇÃO DO USUÁRIO
# ============================================================

# Data da reunião (formato YYYYMMDD)
MEETING_DATE = '20260319'

# Resumo em pt-BR (será a primeira seção do relatório)
SUMMARY_TEXT = """
Em decisão esperada, o Comitê manteve a taxa de juros (fed funds rate) inalterada na faixa de 4,25%-4,50%.
"""

# Headlines Bloomberg (bold=True para destaques principais)
HEADLINES = [
    {"text": "*FED HOLDS BENCHMARK RATE AT 4.25%-4.50%, AS EXPECTED", "bold": True},
    {"text": "*FED SAYS UNCERTAINTY AROUND ECONOMIC OUTLOOK HAS INCREASED", "bold": True},
    {"text": "*FED REPEATS IT WILL ASSESS INCOMING DATA IN CONSIDERING RATE PATH", "bold": False},
]

# Headlines da coletiva do Powell (só para reuniões com SEP, ou None)
PRESSER_HEADLINES = [
    {"text": "*POWELL: WE ARE WELL POSITIONED TO WAIT FOR GREATER CLARITY", "bold": True},
    {"text": "*POWELL: ECONOMY IS IN A GOOD PLACE", "bold": False},
]

# Comentários dos bancos (nome → texto)
BANK_COMMENTS = {
    "Barclays": "Sem alteração na comunicação...",
    "BofA": "O Fed manteve a taxa inalterada...",
    "Goldman Sachs": "Decisão em linha com o esperado...",
}

# Screenshots (caminhos para imagens)
# Reação de mercado (Bloomberg, 9 painéis: NASDAQ, SPX, Russell, UST 2Y, 10Y, 2Y/10Y, OIS 1Y3Y, DXY, VIX)
MARKET_REACTION_PNG = r"c:\Users\mmart\OneDrive\BCB\dirin\15_informes\fomc\screenshots\market_reaction.png"

# Dot plot screenshot (só SEP, ou None para reuniões sem SEP)
DOT_PLOT_PNG = r"c:\Users\mmart\OneDrive\BCB\dirin\15_informes\fomc\screenshots\dot_plot.png"

# Comentário da Mesa de Investimentos (opcional)
MESA_COMMENT = ""

# ============================================================
print(f"Reunião configurada: {MEETING_DATE}")
print(f"Tipo: {'COM SEP' if is_sep_meeting(MEETING_DATE) else 'SEM SEP'}")

Reunião configurada: 20260319
Tipo: COM SEP


In [3]:
# Download automático de documentos do Fed
print("Verificando documentos disponíveis...\n")

updates = check_for_updates()
for doc_type, has_update in updates.items():
    status = "NOVO disponível" if has_update else "Atualizado"
    print(f"  {doc_type}: {status}")

# Download de documentos faltantes
if any(updates.values()):
    print("\nBaixando documentos novos...")
    downloaded = fetch_latest_documents()
    for doc_type, path in downloaded.items():
        if path:
            print(f"  ✓ {doc_type}: {path.name}")
else:
    print("\nTodos os documentos estão atualizados.")

# Listar documentos da reunião
documents = get_meeting_documents(MEETING_DATE)
print(f"\nDocumentos para reunião {MEETING_DATE}:")
for doc_type, path in documents.items():
    status = f"✓ {path.name}" if path else "✗ Não encontrado"
    print(f"  {doc_type}: {status}")

Verificando documentos disponíveis...

  statement: Atualizado
  minutes: Atualizado
  projections: Atualizado
  presser: Atualizado

Todos os documentos estão atualizados.

Documentos para reunião 20260319:
  statement: ✗ Não encontrado
  minutes: ✗ Não encontrado
  presser: ✗ Não encontrado
  projections: ✗ Não encontrado


In [4]:
# Parse do statement
statement_data = {}
if documents['statement']:
    statement_data = parse_statement(documents['statement'])

    print("=" * 50)
    print("DECISÃO DO FOMC")
    print("=" * 50)
    decision_map = {"hold": "MANUTENÇÃO", "cut": "CORTE", "hike": "ALTA"}
    print(f"Decisão: {decision_map.get(statement_data.get('decision', ''), 'N/A')}")
    print(f"Taxa alvo: {statement_data.get('target_rate_low', 'N/A')}% - {statement_data.get('target_rate_high', 'N/A')}%")
    print(f"Unânime: {'Sim' if statement_data.get('unanimous') else 'Não'}")
    if statement_data.get('dissenters'):
        print(f"Dissidentes: {', '.join(statement_data['dissenters'])}")
    if statement_data.get('key_phrases'):
        print(f"\nFrases-chave:")
        for phrase in statement_data['key_phrases']:
            print(f"  - {phrase}")
    print("=" * 50)
else:
    print("Statement não disponível para esta reunião.")

Statement não disponível para esta reunião.


In [5]:
# Projeções SEP (condicional — só para reuniões Mar/Jun/Set/Dez)
sep_current = None
sep_prior = None
prior_label = None
current_ct = None
prior_ct = None

if is_sep_meeting(MEETING_DATE):
    print("Reunião COM SEP — processando projeções...\n")

    # Parse completo do PDF de projeções
    if documents.get('projections'):
        proj_data = parse_projection_table(documents['projections'])
        sep_current = proj_data.get('medians')
        sep_prior = proj_data.get('prior_medians')
        prior_label = proj_data.get('prior_label', 'Prior projection')
        current_ct = proj_data.get('central_tendency')
        prior_ct = proj_data.get('prior_ct')

        if sep_current is not None and not sep_current.empty:
            print("Projeções atuais (mediana):")
            display(sep_current)

        if sep_prior is not None and not sep_prior.empty:
            print(f"\nProjeções anteriores ({prior_label}):")
            display(sep_prior)

        if current_ct is not None and not current_ct.empty:
            print("\nTendência Central (atual):")
            display(current_ct)
    else:
        # Fallback: carregar projeções de fontes alternativas
        prior_date = get_prior_sep_meeting_date(MEETING_DATE)
        if prior_date:
            prior_label = get_prior_meeting_label(prior_date)
            print(f"Carregando projeções anteriores ({prior_label}, {prior_date})...")
            sep_prior_fallback = get_prior_sep_projections(MEETING_DATE)
            if sep_prior_fallback is not None and not sep_prior_fallback.empty:
                sep_prior = sep_prior_fallback
                print("Projeções anteriores carregadas (fallback):")
                display(sep_prior)

    print(f"\nAnos disponíveis: {proj_data.get('year_columns', [])}" if documents.get('projections') else "")
else:
    print("Reunião SEM SEP — seção de projeções será omitida no relatório.")

Reunião COM SEP — processando projeções...

Carregando projeções anteriores (December projection, 20251210)...
Projeções anteriores carregadas (fallback):


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23
0,NaN,Summary of Economic Projections,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Variable,Mediana,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Change in real GDP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Unemployment rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,PCE inflation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Evolução histórica dos Dots (Bloomberg) + Auto-save SEP.xlsx
if is_sep_meeting(MEETING_DATE):
    # --- DOTS Bloomberg ---
    dots_hist = None
    try:
        dots_hist = load_dots_history()
        print("Evolução histórica dos DOTS Medians (Bloomberg):")
        display(dots_hist.tail(10))
    except RuntimeError as e:
        print(f"Bloomberg não disponível para DOTS: {e}")
        try:
            dots_snap = load_dots_snapshot()
            if dots_snap:
                print("DOTS Snapshot (grid1.xlsx):")
                for ticker, val in dots_snap.items():
                    print(f"  {ticker}: {val}")
            else:
                print("Nenhum dado de DOTS disponível.")
        except RuntimeError:
            print("grid1.xlsx não encontrado.")

    # --- Auto-save SEP.xlsx ---
    if sep_current is not None and not sep_current.empty:
        try:
            sep_path = save_sep_to_excel(
                MEETING_DATE, sep_current, sep_prior, prior_label or "Prior projection",
                ct=current_ct, ranges=proj_data.get('range') if documents.get('projections') else None,
            )
            print(f"\nSEP salvo em: {sep_path}")
        except Exception as e:
            print(f"Erro ao salvar SEP.xlsx: {e}")

Evolução histórica dos DOTS Medians (Bloomberg):


,Dots Median Longer Run,Dots Median Ano Corrente,Dots Median Ano+1,Dots Median Ano+2
date,,,,
2023-12-13,2.500,5.375,4.625,3.625
2024-03-20,2.563,4.625,3.875,3.125
2024-06-12,2.750,5.125,4.125,3.125
2024-09-18,2.875,4.375,3.375,2.875
2024-12-18,3.000,4.375,3.875,3.375
2025-03-19,3.000,3.875,3.375,3.125
2025-06-18,3.000,3.875,3.625,3.375
2025-09-17,3.000,3.625,3.375,3.125
2025-12-10,3.000,3.625,3.375,3.125


In [7]:
# Preview do relatório
is_sep = is_sep_meeting(MEETING_DATE)

print("=" * 60)
print("PREVIEW DO RELATÓRIO")
print("=" * 60)

dt = pd.to_datetime(MEETING_DATE, format='%Y%m%d')
print(f"\n📄 Mesa de Investimentos / Depin")
print(f"   Reunião do FOMC – {dt.strftime('%d/%m/%Y')}")
print(f"   Tipo: {'COM SEP (6-10 páginas)' if is_sep else 'SEM SEP (2-4 páginas)'}")

print(f"\n--- CAPA ---")
print(f"  Resumo: {len(SUMMARY_TEXT.split())} palavras")
print(f"  Headlines: {len(HEADLINES)} itens ({sum(1 for h in HEADLINES if h['bold'])} em destaque)")

if is_sep:
    print(f"\n--- SEÇÕES SEP ---")
    print(f"  Dot plot: {'✓' if DOT_PLOT_PNG and Path(DOT_PLOT_PNG).exists() else '✗ (screenshot necessário)'}")
    print(f"  Tabela SEP: {'✓ Atual + Anterior' if sep_prior is not None else '✓ Apenas atual' if sep_current is not None else '✗'}")
    if PRESSER_HEADLINES:
        print(f"  Coletiva Powell: {len(PRESSER_HEADLINES)} headlines")

print(f"\n--- REAÇÃO DE MERCADO ---")
print(f"  Screenshot: {'✓' if Path(MARKET_REACTION_PNG).exists() else '✗ (screenshot necessário)'}")

print(f"\n--- COMENTÁRIOS DOS BANCOS ---")
for bank in BANK_COMMENTS:
    print(f"  • {bank}")

if MESA_COMMENT:
    print(f"\n--- MESA DE INVESTIMENTOS ---")
    print(f"  Comentário: {len(MESA_COMMENT.split())} palavras")

print("\n" + "=" * 60)

PREVIEW DO RELATÓRIO

📄 Mesa de Investimentos / Depin
   Reunião do FOMC – 19/03/2026
   Tipo: COM SEP (6-10 páginas)

--- CAPA ---
  Resumo: 18 palavras
  Headlines: 3 itens (2 em destaque)

--- SEÇÕES SEP ---
  Dot plot: ✗ (screenshot necessário)
  Tabela SEP: ✓ Atual + Anterior
  Coletiva Powell: 2 headlines

--- REAÇÃO DE MERCADO ---
  Screenshot: ✗ (screenshot necessário)

--- COMENTÁRIOS DOS BANCOS ---
  • Barclays
  • BofA
  • Goldman Sachs



In [8]:
# GERAR RELATÓRIO WORD
output_dir = project_root / 'output' / 'reports' / 'fomc'
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"Mesa de Investimentos - FOMC_{MEETING_DATE}.docx"

result_path = generate_fomc_report(
    meeting_date=MEETING_DATE,
    summary_text=SUMMARY_TEXT,
    headlines=HEADLINES,
    market_reaction_path=MARKET_REACTION_PNG,
    bank_comments=BANK_COMMENTS,
    # SEP (None se não for reunião SEP)
    dot_plot_path=DOT_PLOT_PNG if is_sep_meeting(MEETING_DATE) else None,
    sep_current=sep_current,
    sep_prior=sep_prior,
    prior_meeting_label=prior_label,
    presser_headlines=PRESSER_HEADLINES if is_sep_meeting(MEETING_DATE) else None,
    # Central Tendency
    current_ct=current_ct,
    prior_ct=prior_ct,
    include_ct=True,
    mesa_comment=MESA_COMMENT,
    output_path=output_path,
)

print(f"Relatório gerado com sucesso!")
print(f"Arquivo: {result_path}")

Imagem não encontrada: c:\Users\mmart\OneDrive\BCB\dirin\15_informes\fomc\screenshots\market_reaction.png


Relatório gerado com sucesso!
Arquivo: e:\Github\marc3lom\py-bcb\output\reports\fomc\Mesa de Investimentos - FOMC_20260319.docx
